# Toy hierarchy — §4.1 / fig. 2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/PyDeepGP/blob/main/examples/damianou2013/notebooks/01_toy_hierarchy.ipynb)

Port of Matlab [`demToyUnsupervised.m`](https://github.com/SheffieldML/deepGP/blob/master/deepGP/matlab/demToyUnsupervised.m) using
`hgplvmSampleDataTr1.mat`: two observation views `YA`, `YB` share a parent latent
hierarchy. We fit a two-view DeepGP (Q={6,4}) and compare stacked PCA / Isomap.

Paper: [Damianou & Lawrence, AISTATS 2013](https://proceedings.mlr.press/v31/damianou13a.html).


## Setup


In [ ]:
import sys, subprocess, os, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd().resolve()
# Prefer sibling of notebooks/ (repo layout)
for c in [ROOT.parent, ROOT, ROOT / "examples" / "damianou2013"]:
    if (c / "_common.py").is_file():
        ROOT = c
        break

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "numpy", "scipy", "matplotlib", "GPy>=1.15", "scikit-learn"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        f"git+https://github.com/SheffieldML/PyDeepGP.git@examples-damianou2013"])
    ROOT = Path.cwd().resolve()
    (ROOT / "data").mkdir(exist_ok=True)
    for dest, url in [
        ("_common.py", f"https://raw.githubusercontent.com/SheffieldML/PyDeepGP/examples-damianou2013/examples/damianou2013/_common.py"),
        ("_matlab_data.py", f"https://raw.githubusercontent.com/SheffieldML/PyDeepGP/examples-damianou2013/examples/damianou2013/_matlab_data.py"),
        ("data/hgplvmSampleDataTr1.mat",
         "https://raw.githubusercontent.com/SheffieldML/deepGP/master/deepGP/matlab/hgplvmSampleDataTr1.mat"),
        ("data/demHighFiveHgplvm1.mat",
         "https://raw.githubusercontent.com/SheffieldML/deepGP/master/deepGP/matlab/demHighFiveHgplvm1.mat"),
    ]:
        p = ROOT / dest
        if not p.is_file():
            print("Fetching", dest)
            urllib.request.urlretrieve(url, str(p))

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Working directory:", ROOT)


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import deepgp
from _common import (
    ard_weights, dominant_dims, init_noise, unfix_noise, rbf_bias, optimize_staged,
)
from _matlab_data import load_toy_hierarchy


In [ ]:
QUICK = True  # set False for Matlab-demo scale (Q={6,4}, longer optimisation)
q_shared, q_top = (4, 3) if QUICK else (6, 4)
num_inducing = 15 if QUICK else 25
iters_fixed = 50 if QUICK else 200
iters_free = 150 if QUICK else 2000


## Load Matlab ground-truth hierarchy


In [ ]:
data = load_toy_hierarchy()
YA, YB = data["YA"], data["YB"]
X2, XA, XB = data["X2"], data["XA"], data["XB"]
print(f"YA={YA.shape} YB={YB.shape} XA={XA.shape} XB={XB.shape} X2={X2.shape}")

fig, axes = plt.subplots(3, 2, figsize=(9, 8))
axes[0, 0].plot(X2); axes[0, 0].set_title("True top X2")
axes[0, 1].axis("off")
axes[1, 0].plot(XA); axes[1, 0].set_title("True XA")
axes[1, 1].plot(XB); axes[1, 1].set_title("True XB")
axes[2, 0].plot(YA); axes[2, 0].set_title("YA")
axes[2, 1].plot(YB); axes[2, 1].set_title("YB")
fig.suptitle("Matlab hgplvmSampleDataTr1 ground truth")
fig.tight_layout()
plt.show()


## Fit two-view DeepGP (Q shared → top)


In [ ]:
kernels = [[rbf_bias(q_shared), rbf_bias(q_shared)], rbf_bias(q_top)]
m = deepgp.DeepGP(
    [[YA.shape[1], YB.shape[1]], q_shared, q_top],
    [YA, YB],
    kernels=kernels,
    num_inducing=num_inducing,
    back_constraint=False,
    inits="PCA",
)
optimize_staged(m, iters_fixed, iters_free, messages=False)
print(f"variational bound: {float(m.log_likelihood()):.3f}")
for i, view in enumerate(m.layers[0].views):
    print(f"View {i} ARD:", np.array2string(ard_weights(view.kern), precision=3))
print("Top ARD:", np.array2string(ard_weights(m.layers[-1].kern), precision=3))


## Recovered latents (dominant ARD dims)


In [ ]:
Xshare = np.asarray(m.layers[0].X.mean)
Xtop = np.asarray(m.layers[-1].X.mean)
d_top = dominant_dims(ard_weights(m.layers[-1].kern), k=min(2, Xtop.shape[1]))
d_a = dominant_dims(ard_weights(m.layers[0].views[0].kern), k=min(2, Xshare.shape[1]))
d_b = dominant_dims(ard_weights(m.layers[0].views[1].kern), k=min(2, Xshare.shape[1]))

fig, axes = plt.subplots(2, 2, figsize=(8, 7))
axes[0, 0].plot(Xtop[:, d_top]); axes[0, 0].set_title("DeepGP top (dominant dims)")
axes[0, 1].axis("off")
axes[1, 0].plot(Xshare[:, d_a]); axes[1, 0].set_title("shared (view0 ARD dims)")
axes[1, 1].plot(Xshare[:, d_b]); axes[1, 1].set_title("shared (view1 ARD dims)")
fig.suptitle("demToyUnsupervised-style DeepGP recovery")
fig.tight_layout()
plt.show()


## Baselines: stacked PCA and Isomap


In [ ]:
def stacked_pca(Ya, Yb, q1, q2):
    def pca(Y, q):
        Yc = Y - Y.mean(0)
        U, S, _ = np.linalg.svd(Yc, full_matrices=False)
        q = min(q, U.shape[1])
        return U[:, :q] * S[:q]
    xa, xb = pca(Ya, q1), pca(Yb, q1)
    return xa, xb, pca(np.hstack([xa, xb]), q2)

pca_xa, pca_xb, pca_x2 = stacked_pca(YA, YB, q_shared, q_top)
fig, axes = plt.subplots(2, 2, figsize=(8, 7))
axes[0, 0].plot(pca_x2[:, :min(2, pca_x2.shape[1])]); axes[0, 0].set_title("Stacked PCA top")
axes[0, 1].axis("off")
axes[1, 0].plot(pca_xa[:, :min(2, pca_xa.shape[1])]); axes[1, 0].set_title("Stacked PCA XA")
axes[1, 1].plot(pca_xb[:, :min(2, pca_xb.shape[1])]); axes[1, 1].set_title("Stacked PCA XB")
fig.suptitle("Stacked PCA")
fig.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import Isomap

def stacked_isomap(Ya, Yb, q1, q2, n_neighbors=5):
    xa = Isomap(n_components=min(q1, Ya.shape[0] - 1), n_neighbors=n_neighbors).fit_transform(Ya)
    xb = Isomap(n_components=min(q1, Yb.shape[0] - 1), n_neighbors=n_neighbors).fit_transform(Yb)
    x2 = Isomap(n_components=min(q2, xa.shape[0] - 1), n_neighbors=n_neighbors).fit_transform(
        np.hstack([xa, xb])
    )
    return xa, xb, x2

iso_xa, iso_xb, iso_x2 = stacked_isomap(YA, YB, min(2, q_shared), min(2, q_top))
fig, axes = plt.subplots(2, 2, figsize=(8, 7))
axes[0, 0].plot(iso_x2); axes[0, 0].set_title("Stacked Isomap top")
axes[0, 1].axis("off")
axes[1, 0].plot(iso_xa); axes[1, 0].set_title("Stacked Isomap XA")
axes[1, 1].plot(iso_xb); axes[1, 1].set_title("Stacked Isomap XB")
fig.suptitle("Stacked Isomap")
fig.tight_layout()
plt.show()
